## Step 1 — Install Dependencies

In [1]:
!pip install flask flask-cors requests openpyxl numpy pandas scikit-learn --quiet

## Step 2 — HubSpot Access Token

In [2]:
import os
HUBSPOT_ACCESS_TOKEN = os.environ.get("HUBSPOT_ACCESS_TOKEN", "pat-na2-2a5219af-71c8-4fac-a898-ca9cbdd4993f")
HUBSPOT_ACCESS_TOKEN = HUBSPOT_ACCESS_TOKEN.strip()
IS_CONFIGURED = bool(HUBSPOT_ACCESS_TOKEN and HUBSPOT_ACCESS_TOKEN != "ENTER_YOUR_TOKEN_HERE")
print("HubSpot configured:", IS_CONFIGURED)

HubSpot configured: True


## Step 3 — Config & Constants

In [3]:
import re, os
HUBSPOT_API_BASE   = "https://api.hubapi.com"
EXCEL_FILE_PATH    = os.path.join(os.getcwd(), "billswift_registrations.xlsx")
TRAINING_FILE_PATH = os.path.join(os.getcwd(), "billswift_testing_dataset.xlsx")
TARGET_SHEET_NAME  = "Registrations"
TABLE_NAME         = "Table1"
FREE_EMAIL_DOMAINS = {
    "gmail.com","yahoo.com","hotmail.com","outlook.com","aol.com",
    "icloud.com","live.com","mail.com","protonmail.com","rediffmail.com",
    "yandex.com","zoho.com","gmx.com",
}
REQUIRED_FIELDS = [
    "fullName","workEmail","companyName","industry","companySize",
    "transactionVolume","revenueChallenge","currentBilling","heardFrom",
]
EXCEL_HEADERS = [
    "Timestamp","Name","Email","Company Name","Company Domain",
    "Industry","Industry Other","Company Size",
    "Monthly Transaction Volume","Primary Revenue Challenge",
    "Current Billing Solution","Heard From",
    "Lead Score","Lead Tier",
    "HubSpot Contact ID","HubSpot Company ID","Predicted Revenue",
]
COMPANY_SIZE_MAP = {"1-10":"10","11-50":"50","51-200":"200","201-1000":"1000","1000+":"5000"}
print("Config loaded.")
print("Excel :", EXCEL_FILE_PATH)
print("Train :", TRAINING_FILE_PATH)

Config loaded.
Excel : /Users/praveen/Desktop/Job Related/Praveen Project/Python Files/billswift_registrations.xlsx
Train : /Users/praveen/Desktop/Job Related/Praveen Project/Python Files/billswift_testing_dataset.xlsx


## Step 4 — Data Cleaning

In [4]:
def clean_name(v): return v.strip().title()
def clean_data(data):
    c = {k: v.strip() if isinstance(v,str) else v for k,v in data.items()}
    if c.get("fullName"):    c["fullName"]    = clean_name(c["fullName"])
    if c.get("companyName"): c["companyName"] = clean_name(c["companyName"])
    if c.get("workEmail"):   c["workEmail"]   = c["workEmail"].lower()
    return c
def is_valid_email(e): return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", e))
def derive_domain(email):
    if "@" not in email: return ""
    d = email.split("@")[-1].strip().lower()
    return "Personal Email" if d in FREE_EMAIL_DOMAINS else d
def split_name(n):
    p = n.strip().split(" ",1); return p[0],(p[1] if len(p)>1 else "")
print("Data cleaning ready.")
print("Demo:", clean_name("jOHN dOE"), "|", derive_domain("test@gmail.com"))

Data cleaning ready.
Demo: John Doe | Personal Email


## Step 5 — Vectorized Lead Scoring

| Signal | Max |
|--------|-----|
| Monthly Transaction Volume | 28 |
| Primary Revenue Challenge | 24 |
| Current Billing Solution | 19 |
| Company Size | 14 |
| How You Heard About Us | 10 |
| Email Type | 5 |

Tiers: 0-35 Cold · 36-65 Warm · 66-100 Hot

In [5]:
import numpy as np
VOLUME_VALUES    = np.array(["under-50k","50k-250k","250k-1m","1m-10m","10m+"])
VOLUME_SCORES    = np.array([2,7,14,22,28])
CHALLENGE_VALUES = np.array(["research","manual-invoicing","usage-metering","rev-recognition","failed-payments"])
CHALLENGE_SCORES = np.array([3,13,24,17,13])
BILLING_VALUES   = np.array(["modern-platform","legacy-enterprise","in-house","stripe-paypal","none-manual"])
BILLING_SCORES   = np.array([3,9,13,16,19])
SIZE_VALUES      = np.array(["1-10","11-50","51-200","201-1000","1000+"])
SIZE_SCORES      = np.array([2,4,7,11,14])
HEARD_VALUES     = np.array(["other","google","newsletter","linkedin","partner","referral"])
HEARD_SCORES     = np.array([2,4,5,6,8,10])
EMAIL_VALUES     = np.array(["work","personal"])
EMAIL_SCORES     = np.array([5,0])
def get_email_type(email):
    d = email.split("@")[-1].strip().lower() if "@" in email else ""
    return "personal" if d in FREE_EMAIL_DOMAINS else "work"
def vscore(val, vals, scores): return int(np.where(vals==val, scores, 0).sum())
def calculate_lead_score(data):
    et = get_email_type(data.get("workEmail",""))
    total = (vscore(data.get("transactionVolume",""),VOLUME_VALUES,VOLUME_SCORES)
           + vscore(data.get("revenueChallenge",""),CHALLENGE_VALUES,CHALLENGE_SCORES)
           + vscore(data.get("currentBilling",""),BILLING_VALUES,BILLING_SCORES)
           + vscore(data.get("companySize",""),SIZE_VALUES,SIZE_SCORES)
           + vscore(data.get("heardFrom",""),HEARD_VALUES,HEARD_SCORES)
           + vscore(et,EMAIL_VALUES,EMAIL_SCORES))
    tier = "Hot" if total>=66 else ("Warm" if total>=36 else "Cold")
    return total, tier
print("Lead scoring ready. Max: 100 pts.")

Lead scoring ready. Max: 100 pts.


## Step 6 — ML Revenue Prediction Engine

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

_model = None
_model_columns = None

def train_model():
    global _model, _model_columns
    if not os.path.exists(TRAINING_FILE_PATH):
        print("WARNING: Training file not found. Revenue prediction disabled.")
        return None, None
    df = pd.read_excel(TRAINING_FILE_PATH, engine="openpyxl")
    FEAT = ["Industry","Company Size","Monthly Transaction Volume",
            "Primary Revenue Challenge","Current Billing Solution","Heard From","Lead Score"]
    X = pd.get_dummies(df[FEAT].fillna("missing"), columns=FEAT[:-1])
    y = df["Annual Revenue"]
    m = RandomForestRegressor(n_estimators=100, random_state=42)
    m.fit(X, y)
    print(f"Model trained on {len(df)} rows.")
    return m, list(X.columns)

def predict_revenue(data, score):
    global _model, _model_columns
    if _model is None: return None
    FEAT = ["Industry","Company Size","Monthly Transaction Volume",
            "Primary Revenue Challenge","Current Billing Solution","Heard From","Lead Score"]
    row = {"Industry":data.get("industry","missing"),
           "Company Size":data.get("companySize","missing"),
           "Monthly Transaction Volume":data.get("transactionVolume","missing"),
           "Primary Revenue Challenge":data.get("revenueChallenge","missing"),
           "Current Billing Solution":data.get("currentBilling","missing"),
           "Heard From":data.get("heardFrom","missing"),
           "Lead Score":score}
    df_row = pd.DataFrame([row])
    enc = pd.get_dummies(df_row, columns=FEAT[:-1])
    aligned = enc.reindex(columns=_model_columns, fill_value=0)
    return round(float(_model.predict(aligned)[0]), 2)

def ensure_table(ws):
    ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
    if TABLE_NAME in ws.tables:
        ws.tables[TABLE_NAME].ref = ref; return
    for n in list(ws.tables.keys()): del ws.tables[n]
    t = Table(displayName=TABLE_NAME, ref=ref)
    t.tableStyleInfo = TableStyleInfo(name="TableStyleMedium9",
        showFirstColumn=False,showLastColumn=False,showRowStripes=True,showColumnStripes=False)
    ws.add_table(t)

print("ML prediction engine ready.")

ML prediction engine ready.


## Step 7 — Excel Logging

In [7]:
from openpyxl import Workbook, load_workbook

def append_to_excel(row):
    if os.path.exists(EXCEL_FILE_PATH):
        wb = load_workbook(EXCEL_FILE_PATH)
        ws = wb[TARGET_SHEET_NAME] if TARGET_SHEET_NAME in wb.sheetnames else wb.create_sheet(TARGET_SHEET_NAME)
        if ws.max_row==1 and ws.cell(1,1).value is None: ws.append(EXCEL_HEADERS)
    else:
        wb = Workbook(); ws = wb.active
        ws.title = TARGET_SHEET_NAME; ws.append(EXCEL_HEADERS)
    ws.append(row)
    new_row = ws.max_row
    ensure_table(ws)
    wb.save(EXCEL_FILE_PATH)
    return new_row

def update_excel_row(row_num, contact_id, company_id, predicted_revenue):
    wb = load_workbook(EXCEL_FILE_PATH)
    ws = wb[TARGET_SHEET_NAME] if TARGET_SHEET_NAME in wb.sheetnames else wb.active
    if contact_id:       ws.cell(row=row_num, column=EXCEL_HEADERS.index("HubSpot Contact ID")+1).value = contact_id
    if company_id:       ws.cell(row=row_num, column=EXCEL_HEADERS.index("HubSpot Company ID")+1).value = company_id
    if predicted_revenue is not None:
        ws.cell(row=row_num, column=EXCEL_HEADERS.index("Predicted Revenue")+1).value = predicted_revenue
    ensure_table(ws)
    wb.save(EXCEL_FILE_PATH)

print("Excel logging ready. Table1 preserved on every save.")

Excel logging ready. Table1 preserved on every save.


## Step 8 — HubSpot Functions

- 409 conflicts handled automatically — updates existing records instead of failing
- Value translation maps convert form values to HubSpot dropdown labels exactly

In [8]:
import requests

VOLUME_MAP = {
    "under-50k":"Under 50k","50k-250k":"50k - 250k","250k-1m":"250k - 1M",
    "1m-10m":"1M - 10M","10m+":"10M+",
}
CHALLENGE_MAP = {
    "research":"Researching tools / Pre-launch",
    "manual-invoicing":"Manual invoicing overhead",
    "usage-metering":"Usage data metering gaps",
    "rev-recognition":"Complex revenue recognition",
    "failed-payments":"Leaking revenue via failed payments",
}
def translate(val, mapping): return mapping.get(val, val)

def find_company_by_domain(domain):
    h = {"Authorization":f"Bearer {HUBSPOT_ACCESS_TOKEN}","Content-Type":"application/json"}
    r = requests.post(f"{HUBSPOT_API_BASE}/crm/v3/objects/companies/search",
        json={"filterGroups":[{"filters":[{"propertyName":"website","operator":"CONTAINS_TOKEN","value":domain}]}],"limit":1},
        headers=h, timeout=10)
    r.raise_for_status()
    res = r.json().get("results",[])
    return res[0]["id"] if res else None

def upsert_hubspot_company(data, domain, score, tier):
    h = {"Authorization":f"Bearer {HUBSPOT_ACCESS_TOKEN}","Content-Type":"application/json"}
    props = {"name":data.get("companyName",domain),"website":f"https://{domain}",
             "numberofemployees":COMPANY_SIZE_MAP.get(data.get("companySize",""),""),
             "lead_score":str(score),"lead_tier":tier}
    eid = find_company_by_domain(domain)
    if eid:
        requests.patch(f"{HUBSPOT_API_BASE}/crm/v3/objects/companies/{eid}",
            json={"properties":props},headers=h,timeout=10).raise_for_status()
        return eid
    r = requests.post(f"{HUBSPOT_API_BASE}/crm/v3/objects/companies",
        json={"properties":props},headers=h,timeout=10)
    if r.status_code==409:
        eid = find_company_by_domain(domain)
        if eid:
            requests.patch(f"{HUBSPOT_API_BASE}/crm/v3/objects/companies/{eid}",
                json={"properties":props},headers=h,timeout=10).raise_for_status()
            return eid
    r.raise_for_status()
    return r.json()["id"]

def find_contact_by_email(email):
    h = {"Authorization":f"Bearer {HUBSPOT_ACCESS_TOKEN}","Content-Type":"application/json"}
    r = requests.post(f"{HUBSPOT_API_BASE}/crm/v3/objects/contacts/search",
        json={"filterGroups":[{"filters":[{"propertyName":"email","operator":"EQ","value":email}]}],"limit":1},
        headers=h, timeout=10)
    r.raise_for_status()
    res = r.json().get("results",[])
    return res[0]["id"] if res else None

def upsert_hubspot_contact(data, domain, score, tier):
    first, last = split_name(data["fullName"])
    h = {"Authorization":f"Bearer {HUBSPOT_ACCESS_TOKEN}","Content-Type":"application/json"}
    props = {
        "email":data["workEmail"],"firstname":first,"lastname":last,
        "company":data.get("companyName",domain),"website":f"https://{domain}",
        "current_billing_solution":data.get("currentBilling",""),
        "heard_from":data.get("heardFrom",""),
        "lead_score":str(score),"lead_tier":tier,
        "monthly_transaction_volume":translate(data.get("transactionVolume",""),VOLUME_MAP),
        "primary_revenue_challenge":translate(data.get("revenueChallenge",""),CHALLENGE_MAP),
    }
    r = requests.post(f"{HUBSPOT_API_BASE}/crm/v3/objects/contacts",
        json={"properties":props},headers=h,timeout=10)
    if r.status_code==409:
        eid = find_contact_by_email(data["workEmail"])
        if eid:
            requests.patch(f"{HUBSPOT_API_BASE}/crm/v3/objects/contacts/{eid}",
                json={"properties":props},headers=h,timeout=10).raise_for_status()
            return eid
    r.raise_for_status()
    return r.json()["id"]

def associate_contact_company(contact_id, company_id):
    requests.put(
        f"{HUBSPOT_API_BASE}/crm/v3/objects/contacts/{contact_id}/associations/companies/{company_id}/contact_to_company",
        headers={"Authorization":f"Bearer {HUBSPOT_ACCESS_TOKEN}"},timeout=10).raise_for_status()

print("HubSpot upsert functions ready.")
print("Translation maps loaded — dropdown values match HubSpot exactly.")

HubSpot upsert functions ready.
Translation maps loaded — dropdown values match HubSpot exactly.


## Step 9 — Flask App & /api/register Endpoint

On every form submission:
1. Validate fields → 2. Clean data → 3. Score lead → 4. Predict revenue
5. Log to Excel → 6. Upsert HubSpot Contact + Company → 7. Write IDs back to Excel

In [9]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from datetime import datetime, timezone
from openpyxl import load_workbook

app = Flask(__name__)
CORS(app)

@app.route("/api/register", methods=["POST"])
def register():
    if not bool(HUBSPOT_ACCESS_TOKEN and HUBSPOT_ACCESS_TOKEN != "ENTER_YOUR_TOKEN_HERE"):
        return jsonify({"success":False,"error":"Enter a valid HubSpot token."}), 400

    raw = request.get_json(silent=True)
    if not raw: return jsonify({"success":False,"error":"Invalid JSON body."}), 400

    missing = [f for f in REQUIRED_FIELDS if not raw.get(f)]
    if missing: return jsonify({"success":False,"error":f"Missing: {', '.join(missing)}"}), 400

    data   = clean_data(raw)
    email  = data["workEmail"]
    domain = derive_domain(email)

    if not is_valid_email(email):
        return jsonify({"success":False,"error":"Invalid email address."}), 400

    score, tier       = calculate_lead_score(data)
    predicted_revenue = predict_revenue(data, score)

    industry_label = data.get("industry","")
    if industry_label == "other": industry_label = data.get("industryOther","other")

    excel_row = [
        datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        data.get("fullName",""), email, data.get("companyName",""), domain,
        industry_label, data.get("industryOther",""), data.get("companySize",""),
        data.get("transactionVolume",""), data.get("revenueChallenge",""),
        data.get("currentBilling",""), data.get("heardFrom",""),
        score, tier, "", "",
        predicted_revenue if predicted_revenue is not None else "",
    ]
    new_row_num = append_to_excel(excel_row)

    resp = {"success":True,"leadScore":score,"leadTier":tier,
            "predictedRevenue":predicted_revenue,"excelLogged":True,"hubspotSynced":False}

    try:
        company_id = upsert_hubspot_company(data, domain, score, tier)
        contact_id = upsert_hubspot_contact(data, domain, score, tier)
        associate_contact_company(contact_id, company_id)
        update_excel_row(new_row_num, contact_id, company_id, predicted_revenue)
        resp.update({"hubspotSynced":True,"hubspotContactId":contact_id,"hubspotCompanyId":company_id})
    except requests.HTTPError as e:
        err = (e.response.json() if e.response is not None else {}).get("message",str(e))
        resp["hubspotError"] = err
        print(f"HubSpot CRM Sync warning: {getattr(e.response,'status_code','')} {err}")
    except requests.RequestException as e:
        resp["hubspotError"] = str(e)

    return jsonify(resp)

@app.route("/health", methods=["GET"])
def health():
    ok = bool(HUBSPOT_ACCESS_TOKEN and HUBSPOT_ACCESS_TOKEN != "ENTER_YOUR_TOKEN_HERE")
    return jsonify({"status":"ok" if ok else "missing_token",
                    "hubspotConfigured":ok,"excelFile":EXCEL_FILE_PATH,"modelLoaded":_model is not None})

print("Flask app defined.")

Flask app defined.


## Step 10 — Train Model & Start Server

Trains the ML model then starts Flask in a background thread.
If you see **Address already in use** change `PORT = 5051` to `5052`, restart kernel, re-run all cells.

In [10]:
import threading, time

PORT = 5051

_model, _model_columns = train_model()

def run_app():
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_app, daemon=True)
server_thread.start()
time.sleep(1.5)

print(f"Server is live at http://localhost:{PORT}")
print(f"API_BASE_URL in your HTML file matches exactly: http://localhost:{PORT}")
print(f"Model loaded: {_model is not None}")

Model trained on 200 rows.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5051
 * Running on http://192.168.1.23:5051
Press CTRL+C to quit


Server is live at http://localhost:5051
API_BASE_URL in your HTML file matches exactly: http://localhost:5051
Model loaded: True


127.0.0.1 - - [02/Jul/2026 01:11:14] "OPTIONS /api/register HTTP/1.1" 200 -
127.0.0.1 - - [02/Jul/2026 01:11:18] "POST /api/register HTTP/1.1" 200 -


## Step 11 — Inspect the Excel File

Run anytime to check what has been logged.

In [11]:
from openpyxl import load_workbook

if os.path.exists(EXCEL_FILE_PATH):
    wb = load_workbook(EXCEL_FILE_PATH)
    ws = wb[TARGET_SHEET_NAME] if TARGET_SHEET_NAME in wb.sheetnames else wb.active
    print(f"Registrations: {ws.max_row-1} | Table: {list(ws.tables.keys())}")
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        label = "HEADER" if i==0 else f"Row {i}"
        print(f"── {label} ──")
        for h, v in zip(EXCEL_HEADERS, row):
            if v not in (None,""): print(f"   {h}: {v}")
else:
    print("Excel file not found — submit a registration first.")

Registrations: 0 | Table: ['Table1']
── HEADER ──
   Timestamp: Timestamp
   Name: Name
   Email: Email
   Company Name: Company Name
   Company Domain: Company Domain
   Industry: Industry
   Industry Other: Industry Other
   Company Size: Company Size
   Monthly Transaction Volume: Monthly Transaction Volume
   Primary Revenue Challenge: Primary Revenue Challenge
   Current Billing Solution: Current Billing Solution
   Heard From: Heard From
   Lead Score: Lead Score
   Lead Tier: Lead Tier
   HubSpot Contact ID: HubSpot Contact ID
   HubSpot Company ID: HubSpot Company ID
   Predicted Revenue: Predicted Revenue
